# austruct — quickstart

Design a reinforced concrete beam end to end, in a notebook.

**Before anything else:** no module in this package has been verified against the
printed standards. Every constant is tagged `[VECTOR]` in the source. The
mechanics are tested; the *numbers taken from the codes* still need a person
with the standards open. See the README.

Run `AUSTRUCT_STRICT=1` to make unverified modules raise instead of returning.

## 1. Setup

In [ ]:
import austruct
from austruct.analysis import (
    UDL,
    LoadTrain,
    PointLoad,
    analyse_combinations,
    influence_line,
    moving_load_envelope,
    simply_supported,
)
from austruct.core.units import kN, kN_per_m, kNm, m
from austruct.design import as3600
from austruct.design_documentation import designate, parse
from austruct.loads import ActionType, FillDispersal, LoadCase
from austruct.materials import describe_options
from austruct.project import Occupancy, Project
from austruct.report import Report, plots

austruct.__version__

## 2. The unit convention

**N, mm, MPa.** Plain floats. Multiply on the way in, divide on the way out.
Note `1 kN/m == 1 N/mm` exactly.

In [ ]:
8 * m, 100 * kN, 25 * kN_per_m, 200 * kNm

## 3. Define the section

The designation grammar is the plain-text record of a design decision. It
round-trips: `parse(designate(s))` reproduces the section.

In [ ]:
section = parse("350 x 650 | C40 | COV 40 | BOT 4-N28 | TOP 2-N16 | LIG N12-2L@200",
                name="B1")
section

In [ ]:
designate(section)   # back to text, unchanged

In [ ]:
plots.plot_section(section)

## 4. Project data → load combinations

The occupancy you state drives the ψ factors, which drive the combinations.
This is the conversion the project record exists to perform.

In [ ]:
project = Project(
    job_number="24-1234", job_name="Riverside Apartments",
    occupancy=Occupancy.OFFICE, engineer="A. Morrison",
)
project

In [ ]:
combos = project.load_combinations(sls=False)
for c in combos:
    print(c)

## 5. Analyse, over every combination

The envelope records **which combination governs at every position** — the part
usually lost.

In [ ]:
beam = simply_supported(8 * m, section=section, name="B1")

cases = (
    LoadCase("G", ActionType.G, (UDL(magnitude=20 * kN_per_m),)),
    LoadCase("Q", ActionType.Q, (UDL(magnitude=15 * kN_per_m),
                                 PointLoad(position=3 * m, magnitude=60 * kN))),
)
env = analyse_combinations(beam, cases, combos)
env

In [ ]:
plots.plot_envelope(env, show_cases=True)

Diagrams and envelopes convert straight to a DataFrame — no pandas dependency in the package:

In [ ]:
import pandas as pd
pd.DataFrame(env.moment.to_table()).iloc[::40].head(8)

## 6. Design checks

Every calculation returns a `CalcResult`, never a bare float — so it carries its
inputs, clause references, validity envelope and provenance with it.

In [ ]:
flexure = as3600.check_flexure(section, env.M_star)
flexure

In [ ]:
shear = as3600.check_shear(section, V_star=env.shear_at_d_from_support(section.d),
                           M_star=env.M_star)
print(f"V* = {env.V_star / kN:.1f} kN   phi.Vu = {shear.get('phiVu') / kN:.1f} kN")
print(f"utilisation {shear.utilisation:.3f}   {'PASS' if shear.passed else 'FAIL'}")

The inverse problem — what steel does this moment actually need?

In [ ]:
required = as3600.required_steel_area(section, env.M_star)
print(describe_options(required.get("Ast_req")))

## 7. Moving loads

A traffic model is a pattern with a *free position*, so the analysis searches
for the worst one. The governing "case" label becomes the governing position.

In [ ]:
train = LoadTrain("2-axle", axles=((0.0, 100 * kN), (4 * m, 100 * kN)), length=4 * m)
result = moving_load_envelope(beam, train, step=100.0)
result

In [ ]:
print(f"M*    = {result.M_star / kNm:.1f} kN.m")
print(f"exact = {100 * kN * (2 * 8 * m - 4 * m) ** 2 / (8 * 8 * m) / kNm:.1f} kN.m  (P(2L-a)^2/8L)")
print(f"leading axle at x = {result.critical_position('moment') / 1000:.3f} m")

### Nominating a standard load model

You should never restate an axle spacing. Name the model and it carries its own
geometry — axles, wheels, contact patch, lane UDL and dynamic allowance.

The AS 5100.2 geometry in this package is **UNVERIFIED**, so the catalogue
refuses to hand out a model until permission is given. Listing it needs no
permission: seeing what exists is not using it.

In [ ]:
from austruct.loads import as5100_2 as traffic

print(traffic.catalogue())

In [ ]:
traffic.allow_unverified(True)   # once per session — development only

m1600 = traffic.get("M1600")
m1600

In [ ]:
# Everything the model carries, so no call site has to restate it.
print(f"axle load  = {m1600.axle_load / kN:.0f} kN")
print(f"wheel load = {m1600.wheel_load / kN:.0f} kN  ({m1600.wheel.n_per_axle} per axle)")
print(f"wheel gap  = {m1600.wheel.spacing:.0f} mm")
print(f"contact    = {m1600.wheel.contact_length:.0f} x {m1600.wheel.contact_width:.0f} mm")

In [ ]:
# Sweep the vehicle, with its dynamic load allowance, over a 20 m bridge span.
bridge = simply_supported(20 * m, EI=1e15, name="BR1")
moving_load_envelope(bridge, m1600.train_with_dla(), step=500.0)

Two deliberate refusals live in here. `300LA` has no *constant* dynamic load
allowance — AS 5100.2 Section 9 makes it a function of loaded length — so asking
for one raises rather than quietly applying a road number:

In [ ]:
try:
    traffic.get("300LA").train_with_dla()
except NotImplementedError as exc:
    print(str(exc).splitlines()[0])

# And the HLP entries admit they are placeholders.
print("HLP320 is a placeholder:", traffic.get("HLP320").is_placeholder)

Influence lines answer the complementary question — *where should the load go*:

In [ ]:
il = influence_line(beam, "moment", location=4 * m, n_points=61)
plots.plot_influence_line(il)

In [ ]:
w = 10 * kN_per_m
print(f"UDL effect via IL area : {w * il.area / kNm:.2f} kN.m")
print(f"exact wL^2/8           : {w * (8 * m) ** 2 / 8 / kNm:.2f} kN.m")

## 8. Buried structure — load through fill

Shallow fill governs the wheel, deep fill governs the earth pressure.

In [ ]:
slab = simply_supported(6 * m, EI=1e14, name="Culvert top slab")
wheel, cl, cw = 80 * kN, 250.0, 400.0

rows = []
for depth in (300, 600, 1200, 2000, 3000):
    fill = FillDispersal(depth=depth, density=2000, slope=2.0, effective_width=1000)
    m_earth = slab.with_loads((fill.earth_pressure_udl(),)).solve().max_moment
    patch = fill.disperse_wheel(wheel, 3 * m, cl, cw, 6 * m)
    m_wheel = slab.with_loads((patch,)).solve().max_moment
    rows.append({"fill_mm": depth,
                 "earth_kPa": fill.vertical_pressure * 1e3,
                 "M_earth_kNm": m_earth / kNm,
                 "M_wheel_kNm": m_wheel / kNm,
                 "M_total_kNm": (m_earth + m_wheel) / kNm})
pd.DataFrame(rows)

Pass the **model** and no geometry is restated at all — the dispersal reads the
contact patch and the wheel spacing off it.

Multi-wheel axles need transverse bookkeeping, and the obvious default is the
wrong one: a 1 m strip on the *centreline* of a 2 m axle under shallow fill
carries nothing, because both patches sit either side of it. So `strip_offset`
defaults to `"worst"`.

In [ ]:
a160 = traffic.get("A160")
rows = []
for depth in (300, 600, 1200, 3000):
    f = FillDispersal(depth=depth, density=2000, slope=2.0, effective_width=1000)
    centre = sum(p.total() for p in f.disperse_model(a160, 3 * m, 6 * m, strip_offset=0.0))
    worst = sum(p.total() for p in f.disperse_model(a160, 3 * m, 6 * m))
    rows.append({"fill_mm": depth,
                 "centreline_kN": centre / kN,
                 "worst_offset_mm": f.worst_strip_offset(a160),
                 "worst_strip_kN": worst / kN})
pd.DataFrame(rows)

In [ ]:
# The dispersed model is still positionable, so it can be swept like any train.
fill = FillDispersal(depth=600, density=2000, slope=2.0, effective_width=1000)
moving_load_envelope(slab, fill.dispersed_model_train(a160), step=100.0,
                     static_loads=(fill.earth_pressure_udl(),))

## 9. The audit report

The layout is fixed: inputs → basis & envelope → working → checks → figures →
signatures. Renderers are interchangeable.

In [ ]:
report = Report(
    title="Beam B1 — flexural and shear design",
    signature=project.signature_block(element="Beam B1", revision="A"),
)
report.add(env.to_calc_result())
report.add(flexure)
report.add(shear)

fig = plots.plot_diagrams(beam.with_loads(
    tuple(load for case in cases for load in case.loads)).solve())
report.add_figure("Diagrams (unfactored G + Q)", plots.save(fig, "b1_diagrams.png"))

print(f"passed:   {report.passed}")
print(f"issuable: {report.issuable}   <- no module is verified yet")

In [ ]:
report

## 10. What is built, and what is verified

Two different questions, answered by two different reports.

In [ ]:
from austruct.core.registry import REGISTRY
print(REGISTRY.coverage())

In [ ]:
print(REGISTRY.summary())

In [ ]:
from austruct.materials import _data
print(_data.data_verification_report())